# Optimization Module Test Notebook

This notebook tests the **Optimization** module functionality.

**Test Coverage:**
- OptimizationManager initialization and configuration loading
- Single-objective optimization
- Multi-objective optimization
- Parameter bounds handling
- Performance comparison (baseline vs optimized)
- Optimized parameter saving/loading
- Edge cases and error handling

## 1. Setup and Imports

In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController
from animakinetica.analysis.optimization import OptimizationManager, list_opt_configs

print("✓ Imports successful")

✓ Imports successful


## 2. List Available Optimization Configurations

In [2]:
# List all available optimization configurations
available_configs = list_opt_configs()

print(f"✓ Found {len(available_configs)} optimization configurations")
print(f"\nFirst 10 configurations:")
for config in available_configs[:10]:
    print(f"  - {config}")

✓ Found 2 optimization configurations

First 10 configurations:
  - BSF_optimization
  - default_optimization


## 3. Initialize Baseline Simulation

Create a baseline simulation that will be optimized.

In [3]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])# "fed_batch", "perfusion"])

# Initialize baseline simulation with FEDBATCH mode (faster for testing)
baseline_sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    stoichiometry_mode=StoichiometryMode.CONSTANT,
    operating_mode=OperatingMode.BATCH_WITH_STARTUP_FEDBATCH,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    #perfusion_strategy=PerfusionStrategy.CONSTANT,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=True  # Use performance mode for faster optimization
)

# Run baseline simulation
baseline_result = baseline_sim.run_simulation()

print(f"✓ Baseline simulation completed")
print(f"\nBaseline Performance Metrics:")
print(f"  STY: {baseline_sim.data.summary_stats['STY_kg_m3_day']:.4f} kg/m³/day")
print(f"  Total DCM: {baseline_sim.data.projected_totals_stats['total_DCM_out_kg']:.4f} kg")
print(f"  Medium Conversion Efficiency: {baseline_sim.data.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {baseline_sim.data.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {baseline_sim.data.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {baseline_sim.data.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {baseline_sim.data.projected_totals_stats['number_of_batches']}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
run_simulation execution time: 185.844 ms
✓ 

## 4. Initialize OptimizationManager with Default Config

In [4]:
# Create optimization manager with default configuration
opt_manager = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file="BSF_optimization" # None  # Uses default config
)

print(f"✓ OptimizationManager initialized with default config")
print(f"  Configuration file: {opt_manager.opt_config_file}")
print(f"  Number of parameters to optimize: {len(opt_manager.opt_config)}")

# Display optimization parameters
print(f"\nOptimization Parameters:")
for i, (param_name, config) in enumerate(opt_manager.opt_config.items(), 1):
    bounds = config['bounds']
    label = config['label']
    unit = config['unit']
    print(f"  {i}. {param_name}: [{bounds[0]:.4g}, {bounds[1]:.4g}] {unit} ({label})")

Searching specified optimization config: BSF_optimization
Loading optimization config from: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\BSF_optimization.yaml
Parsing...
Parsed and loaded configuration containing 2 optimization parameters.
✓ OptimizationManager initialized with default config
  Configuration file: BSF_optimization
  Number of parameters to optimize: 2

Optimization Parameters:
  1. initial_conditions.C_Glc_init: [1, 100]  (C_Glc_init)
  2. initial_conditions.C_Gln_init: [1, 100]  (C_Gln_init)


## 5. Test Loading Specific Optimization Configuration

Test loading a configuration file for fed-batch with SIMPLE kinetics.

In [5]:
# Look for fed-batch SIMPLE optimization config
fedbatch_configs = [c for c in available_configs if c.startswith('F_E_S')]

if fedbatch_configs:
    test_config = fedbatch_configs[0]
    print(f"Testing with configuration: {test_config}")
    
    # Create optimization manager with specific config
    opt_manager_specific = OptimizationManager(
        simulation=baseline_sim,
        opt_config_file=test_config
    )
    
    print(f"\n✓ OptimizationManager initialized with specific config")
    print(f"  Number of parameters to optimize: {len(opt_manager_specific.opt_config)}")
    
    # Display first 5 parameters
    print(f"\nFirst 5 Optimization Parameters:")
    for i, (param_name, config) in enumerate(list(opt_manager_specific.opt_config.items())[:5], 1):
        bounds = config['bounds']
        label = config['label']
        unit = config['unit']
        print(f"  {i}. {param_name}: [{bounds[0]:.4g}, {bounds[1]:.4g}] {unit} ({label})")
else:
    print("⚠ No fed-batch SIMPLE configurations found, skipping specific config test")

⚠ No fed-batch SIMPLE configurations found, skipping specific config test


## 6. Single-Objective Optimization Test

Optimize for a single objective (STY = Space-Time Yield).

In [6]:
# Run single-objective optimization (small iteration count for testing)
print("Running single-objective optimization for STY...")
print("Note: Using reduced iterations for testing purposes\n")

# Use the default opt_manager for testing
opt_manager.optimize_parameters(
    objectives=["STY_kg_m3_day"],
    weights=[1.0],
    workers=1,  # Single worker for reproducibility
    verbose=True
)

print(f"\n✓ Single-objective optimization completed")
print(f"\nOptimal Parameters Found:")
for param_name, value in opt_manager.optimal_parameters.items():
    config = opt_manager.opt_config[param_name]
    label = config['label']
    unit = config['unit']
    bounds = config['bounds']
    print(f"  {param_name}: {value:.4g} {unit} ({label})")
    print(f"    Bounds: [{bounds[0]:.4g}, {bounds[1]:.4g}]")

Running single-objective optimization for STY...
Note: Using reduced iterations for testing purposes

Starting optimization with objectives: STY_kg_m3_day (maximize, weight: 1.0)
Worker configuration: requested=1, effective=1
Optimizing 2 parameters: ['initial_conditions.C_Glc_init', 'initial_conditions.C_Gln_init']
Using per-simulation baseline normalization scalars:
  STY_kg_m3_day: 0.3276
run_simulation execution time: 189.337 ms
Baseline objective evaluation: feasible (value=-1; within_bounds=True)
Running differential evolution...
run_simulation execution time: 164.164 ms
run_simulation execution time: 170.106 ms
run_simulation execution time: 191.714 ms
run_simulation execution time: 182.279 ms
run_simulation execution time: 229.006 ms
run_simulation execution time: 196.024 ms
run_simulation execution time: 202.771 ms
run_simulation execution time: 223.606 ms
run_simulation execution time: 161.939 ms
run_simulation execution time: 173.886 ms
run_simulation execution time: 212.290

## 7. Performance Comparison: Baseline vs Optimized

In [7]:
# Compare baseline and optimized performance
comparison_df = opt_manager.compare_performance(verbose=True)


=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.695764 kg Glc+Gln+AA/kg DCM    6.191727     1.495963
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  209.646871                   kg  213.796953     4.150082
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1397.645809               kg WCM 1425.313020    27.667211
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.722117            m3/kg DCM    0.637850    -0.084267
   Protein Conversion Efficiency              Gl

## 8. Multi-Objective Optimization Test

Optimize for multiple objectives with weighted sum approach.

In [9]:
# Create new baseline simulation for multi-objective test
baseline_sim_multi = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    stoichiometry_mode=StoichiometryMode.CONSTANT,
    operating_mode=OperatingMode.BATCH,
    #feed_strategy=FeedStrategy.EXPONENTIAL,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=True
)

# Run baseline
baseline_sim_multi.run_simulation()

# Create optimization manager
opt_manager_multi = OptimizationManager(
    simulation=baseline_sim_multi,
    opt_config_file=None
)

print("Running multi-objective optimization...")
print("Objectives: STY (70%), Feed Conversion Efficiency (30%)\n")

# Run multi-objective optimization
opt_manager_multi.optimize_parameters(
    objectives=["STY_kg_m3_day", "feed_conversion_efficiency_kg_kg"],
    weights=[0.7, 0.3],
    workers=1,  # Two workers for testing
    verbose=True
)

print(f"\n✓ Multi-objective optimization completed")

run_simulation execution time: 290.617 ms
No optimization config specified. Using default...
Loading optimization config from: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\default_optimization.yaml
Parsing...
Parsed and loaded configuration containing 2 optimization parameters.
Running multi-objective optimization...
Objectives: STY (70%), Nutrient Use Efficiency (30%)

Starting optimization with objectives: STY_kg_m3_day (maximize, weight: 0.7), feed_conversion_efficiency_kg_kg (minimize, weight: 0.3)
Worker configuration: requested=1, effective=1
Optimizing 2 parameters: ['initial_conditions.C_Glc_init', 'initial_conditions.C_Gln_init']
Using per-simulation baseline normalization scalars:
  STY_kg_m3_day: 0.3495
  feed_conversion_efficiency_kg_kg: 4.852
run_simulation execution time: 172.688 ms
Baseline objective evaluation: feasible (value=-0.4; within_bounds=True)
Running differential evolution...
ru

## 9. Multi-Objective Performance Comparison

In [20]:
# Compare multi-objective results
comparison_multi_df = opt_manager_multi.compare_performance(verbose=True)

# Analyze trade-offs
print("\n=== MULTI-OBJECTIVE TRADE-OFF ANALYSIS ===")
for metric in ['STY_kg_m3_day', 'feed_conversion_efficiency_kg_kg', 'medium_conversion_efficiency_m3_kg']:
    if metric in comparison_multi_df.index:
        baseline = comparison_multi_df.loc[metric, 'Baseline']
        optimized = comparison_multi_df.loc[metric, 'Optimized']
        improvement_pct = (optimized - baseline) / baseline * 100
        print(f"{metric}:")
        print(f"  Change: {improvement_pct:+.2f}%")


=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.852275 kg Glc+Gln+AA/kg DCM    3.781302    -1.070973
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  223.659691                   kg  223.347687    -0.312004
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1491.064606               kg WCM 1488.984579    -2.080027
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.743324            m3/kg DCM    0.677117    -0.066207
   Protein Conversion Efficiency              Gl

# 10. Test Optimization with Engineering Desirability Scoring


In [ ]:
# Test Optimization with Engineering Desirability Scoring
print("Running optimization with Engineering Desirability scoring...")

# Baseline metrics as reference
base_sty = baseline_sim.data.summary_stats['STY_kg_m3_day']
base_med_eff = baseline_sim.data.summary_stats['medium_conversion_efficiency_m3_kg']

print(f"Baseline STY: {base_sty:.4f} kg/m³/day (maximize)")
print(f"Baseline Medium Conversion Efficiency: {base_med_eff:.4f} m³/kg (minimize)")

# Define desirability config:
# - STY (maximize): minimum acceptable is 50% of baseline, target is 150% of baseline
# - medium_conversion_efficiency_m3_kg (minimize): minimum unacceptable is 150% of baseline, target is 70% of baseline
desirability_config = {
    'STY_kg_m3_day': {
        'minimum': base_sty * 0.5,
        'target': base_sty * 1.5,
        'safety_cap': 1.5
    },
    'medium_conversion_efficiency_m3_kg': {
        'minimum': base_med_eff * 1.5,
        'target': base_med_eff * 0.7,
        'safety_cap': 1.5
    }
}

opt_manager_desirability = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file="BSF_optimization"
)

opt_manager_desirability.optimize_parameters(
    objectives=['STY_kg_m3_day', 'medium_conversion_efficiency_m3_kg'],
    weights=[0.6, 0.4],
    desirability_config=desirability_config,
    scoring_profile="test_desirability_profile",
    workers=1,
    verbose=True
)


Running optimization with Engineering Desirability scoring...
Baseline STY: 0.3276 kg/m³/day (maximize)
Baseline Medium Conversion Efficiency: 0.7221 m³/kg (minimize)
Searching specified optimization config: BSF_optimization
Loading optimization config from: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\optimization_config\BSF_optimization.yaml
Parsing...
Parsed and loaded configuration containing 2 optimization parameters.
Starting optimization with objectives: STY_kg_m3_day (maximize, weight: 0.6), medium_conversion_efficiency_m3_kg (minimize, weight: 0.4)
Worker configuration: requested=1, effective=1
Optimizing 2 parameters: ['initial_conditions.C_Glc_init', 'initial_conditions.C_Gln_init']
Using engineering desirability scoring (per-KPI smooth asymptotic caps):
  STY_kg_m3_day (maximize): minimum=0.1638, target=0.4914, safety_cap=1.5
  medium_conversion_efficiency_m3_kg (minimize): minimum=1.083, target=0.5055, safety_c

In [21]:
print("\n✓ Desirability optimization completed")
comparison_desirability_df = opt_manager_desirability.compare_performance(verbose=True)


✓ Desirability optimization completed

=== PERFORMANCE COMPARISON ===
                          Metric                                                                                  Description    Baseline                 Unit   Optimized  Improvement
      Feed Conversion Efficiency         Glucose, glutamine, and amino-acid input per harvested dry biomass; lower is better.    4.695764 kg Glc+Gln+AA/kg DCM    9.962589     5.266826
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  209.646871                   kg  195.031049   -14.615822
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 1397.645809               kg WCM 1300.206996   -97.438813
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.    0.722117            m3/kg DCM    0.622216    -0.099902
   Protei

## 11. Test Optimization Result Storage and Access

In [22]:
# Test that optimization results are properly stored
print("Optimization Result Attributes:")
print(f"  Success: {opt_manager.optimization_result.success}")
print(f"  Message: {opt_manager.optimization_result.message}")
print(f"  Number of iterations: {opt_manager.optimization_result.nit}")
print(f"  Number of function evaluations: {opt_manager.optimization_result.nfev}")
print(f"  Optimal objective value: {-opt_manager.optimization_result.fun:.4f}")  # Negative because minimization

# Verify optimized simulation exists and has data
print(f"\n✓ Optimized simulation verification:")
print(f"  Data object exists: {opt_manager.optimized_simulation.data is not None}")
print(f"  Number of batches: {len(opt_manager.optimized_simulation.data.batches)}")
print(f"  Summary stats available: {len(opt_manager.optimized_simulation.data.summary_stats) > 0}")

Optimization Result Attributes:
  Success: True
  Message: Optimization terminated successfully.
  Number of iterations: 1
  Number of function evaluations: 477
  Optimal objective value: 1.0198

✓ Optimized simulation verification:
  Data object exists: True
  Number of batches: 9
  Summary stats available: True


## 11. Test Parameter Bounds Processing

In [13]:
# Test bounds processing for different bound types
print("Testing Parameter Bounds Processing:")

# Check that all bounds are valid tuples
bounds_valid = True
for param_name, config in opt_manager.opt_config.items():
    bounds = config['bounds']
    if not isinstance(bounds, tuple) or len(bounds) != 2:
        print(f"  ✗ Invalid bounds for {param_name}: {bounds}")
        bounds_valid = False
    elif bounds[0] >= bounds[1]:
        print(f"  ✗ Invalid bounds range for {param_name}: {bounds}")
        bounds_valid = False

if bounds_valid:
    print(f"✓ All {len(opt_manager.opt_config)} parameter bounds are valid")
else:
    print("✗ Some parameter bounds are invalid")

# Display bounds statistics
all_bounds = [config['bounds'] for config in opt_manager.opt_config.values()]
lower_bounds = [b[0] for b in all_bounds]
upper_bounds = [b[1] for b in all_bounds]

print(f"\nBounds Statistics:")
print(f"  Lower bounds range: [{min(lower_bounds):.2e}, {max(lower_bounds):.2e}]")
print(f"  Upper bounds range: [{min(upper_bounds):.2e}, {max(upper_bounds):.2e}]")

Testing Parameter Bounds Processing:
✓ All 2 parameter bounds are valid

Bounds Statistics:
  Lower bounds range: [1.00e+00, 1.00e+00]
  Upper bounds range: [1.00e+02, 1.00e+02]


## 12. Test ObjectiveFunction Class

Test the pickleable objective function wrapper used for parallel optimization.

In [14]:
from animakinetica.analysis.optimization import ObjectiveFunction

# Create an objective function instance
obj_func = ObjectiveFunction(
    baseline_simulation=baseline_sim,
    opt_config=opt_manager.opt_config,
    objectives=["STY_kg_m3_day"],
    weights=[1.0]
)

print("✓ ObjectiveFunction created")

# Test evaluation with baseline parameters
baseline_param_values = []
for param_name in opt_manager.opt_config.keys():
    value = baseline_sim.parameters.get_parameter_value(param_name)
    baseline_param_values.append(value)

baseline_objective_value = obj_func(baseline_param_values)
print(f"\nBaseline objective value: {baseline_objective_value:.4f}")
print(f"  (negative for maximization, should be negative of STY)")

# Verify consistency with simulation result
expected_value = -baseline_sim.data.summary_stats['STY_kg_m3_day']
print(f"  Expected value: {expected_value:.4f}")
difference = abs(baseline_objective_value - expected_value)
if difference < 1e-6:
    print(f"  ✓ Objective function evaluation matches simulation result")
else:
    print(f"  ⚠ Difference: {difference:.4e} (may be due to re-running simulation)")

✓ ObjectiveFunction created
run_simulation execution time: 197.500 ms

Baseline objective value: -0.3276
  (negative for maximization, should be negative of STY)
  Expected value: -0.3276
  ✓ Objective function evaluation matches simulation result


## 13. Test Multi-Objective Weighting

In [25]:
# Test multi-objective function with different weights
objectives_test = ["STY_kg_m3_day", "medium_conversion_efficiency_m3_kg"]
weights_test = [2, 0.5]

obj_func_multi = ObjectiveFunction(
    baseline_simulation=baseline_sim,
    opt_config=opt_manager.opt_config,
    objectives=objectives_test,
    weights=weights_test
)

multi_obj_value = obj_func_multi(baseline_param_values)
print(f"Multi-objective value (weighted sum): {multi_obj_value:.4f}")

# Calculate expected value taking metric direction into account
sty = baseline_sim.data.summary_stats['STY_kg_m3_day']
med_eff = baseline_sim.data.summary_stats['medium_conversion_efficiency_m3_kg']

# STY is maximized (+), medium_conversion_efficiency is minimized (-)
expected_weighted = -(weights_test[0] * sty - weights_test[1] * med_eff)

print(f"  Expected: {expected_weighted:.4f}")
print(f"  Difference: {abs(multi_obj_value - expected_weighted):.4e}")

if abs(multi_obj_value - expected_weighted) < 1e-6:
    print(f"  ✓ Multi-objective weighting correct")
else:
    print(f"  ⚠ Difference may be due to re-running simulation")

run_simulation execution time: 240.767 ms
Multi-objective value (weighted sum): -0.2941
  Expected: -0.2941
  Difference: 0.0000e+00
  ✓ Multi-objective weighting correct


## 14. Test Error Handling

In [24]:
# Test error handling for invalid inputs
print("Testing error handling:")

# Test 1: Mismatched objectives and weights
try:
    opt_manager.optimize_parameters(
        objectives=["STY_kg_m3_day", "medium_efficiency_kg_m3"],
        weights=[1.0],  # Only one weight for two objectives
        workers=1,
        verbose=False
    )
    print("  ✗ Test 1 failed: Should raise ValueError for mismatched objectives/weights")
except ValueError as e:
    print(f"  ✓ Test 1 passed: ValueError raised as expected")
    print(f"    Message: {str(e)}")

# Test 2: Empty objectives list
try:
    opt_manager.optimize_parameters(
        objectives=[],
        weights=[],
        workers=1,
        verbose=False
    )
    print("  ✗ Test 2 failed: Should raise ValueError for empty objectives")
except ValueError as e:
    print(f"  ✓ Test 2 passed: ValueError raised as expected")
    print(f"    Message: {str(e)}")

# Test 3: Optimization manager without parameters
try:
    sim_no_params = CellCultureSimulation(
        parameters=None,
        kinetics=Kinetics.STOICHIOMETRIC,
        operating_mode=OperatingMode.BATCH
    )
    opt_manager_invalid = OptimizationManager(sim_no_params)
    print("  ✗ Test 3 failed: Should raise ValueError for simulation without parameters")
except (ValueError, AttributeError) as e:
    print(f"  ✓ Test 3 passed: Exception raised as expected")
    print(f"    Message: {str(e)[:80]}...")

print("\n✓ Error handling tests completed")

Testing error handling:
  ✓ Test 1 passed: ValueError raised as expected
    Message: Number of objectives (2) must match number of weights (1)
  ✓ Test 2 passed: ValueError raised as expected
    Message: At least one objective must be specified
  ✓ Test 3 passed: Exception raised as expected
    Message: 'NoneType' object has no attribute 'create_mode_adjusted_copy'...

✓ Error handling tests completed
